# ================================================# Trinity TSCP MC — Kaggle Benchmark Task# Brain Zone: Social Brain / TPJ + mPFC# Questions: 1584 MC (4-choice)# Dataset: playra/trinity-cognitive-probes-tscp-mc# Hackathon: Measuring Progress Toward AGI (DeepMind x Kaggle)# ================================================**Social Cognition Probe** — Multiple Choice BenchmarkTests: Theory of mind, pragmatic inference, audience adaptation, social norms

In [ ]:
# Install dependencies
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

In [ ]:
# Imports and configuration
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
import glob

print("Imports successful")

In [ ]:
# Download dataset from Kaggle
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-tscp-mc',
    path='/kaggle/working/datasets/',
    unzip=True
)

In [ ]:
# Load MC data
csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'tscp_mc.csv' in f][0]

df = pd.read_csv(csv_path)
df = df[df['question_type'] == 'mc'].copy()
eval_df = df.rename(columns={"question": "question", "choices": "choices", "answer": "expected_answer"})

print(f"Loaded {len(eval_df)} TSCP MC questions")

In [ ]:
# Structured output schema for MC answers
@dataclass
class MCAnswer:
    answer: str

print("Schema defined")

In [ ]:
# Inner task: evaluate a single MC question
@kbench.task(name="tscp_single_mc", store_task=False)
def tscp_single_mc(llm, question: str, choices: str, expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D)."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[0]
    exp = str(expected_answer).strip().upper()[0]
    return got == exp

print("Inner task registered")

In [ ]:
# Outer benchmark task: run all MC questions and compute accuracy
@kbench.task(name="tscp_mc_benchmark", description="Social Cognition Probe - Multiple Choice")
def tscp_mc_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tscp_single_mc.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res["result"].notna()]
    correct = int(valid["result"].sum())
    total = len(eval_df)
    acc = float(valid["result"].mean())

    kbench.assertions.assert_true(
        True,
        expectation=f"TSCP MC accuracy: {acc:.2%} ({correct}/{total})"
    )
    return acc

print("Outer benchmark task registered")

In [ ]:
# Execute benchmark and submit to leaderboard
run = tscp_mc_benchmark.run()
print(f"\nTSCP MC Accuracy: {run.result:.1%}")
%choose tscp_mc_benchmark